# Módulo de Precificação de Crédito (Amortização)

Na engenharia financeira de sistemas bancários, o cálculo de uma prestação fixa baseada em juros compostos não é uma simples divisão do montante. Ele obedece ao Sistema Francês de Amortização (**Tabela Price**).

Antes de calcular a parcela, precisamos converter a Taxa de Juros Anual ($i_a$) para a Taxa de Juros Mensal Equivalente ($i_m$), pois os pagamentos serão mensais. A equação de taxas equivalentes em juros compostos é:

$$i_m = (1 + i_a)^\frac{1}{12} - 1$$

Onde:

- $i_m$ é a taxa de juros mensal (em formato decimal, não percentual).
- $i_a$ é a taxa de juros anual (em formato decimal. Ex: 13% = 0.13)

Com a taxa mensal em mãos, calculamos o valor da **Prestação Fixa Mensal ($PMT$)** utilizando a fórmula do **Valor Presente de uma Anuidade**:

$$PMT = P \times \frac{i_m \times (1 + i_m)^n}{(1 + i_m)^n - 1}$$

Onde:

- $PMT$ (Payment) é o valor da prestação mensal.
- $P$ (Principal) é o valor total do imóvel a ser financiado.
- $i_m$ é a taxa de juros mensal em formato decimal.
- $n$ é o prazo total de pagamento em meses.

Veja abaixo o código Python que transforma o cálculo matemático em uma função que determina o valor da prestação mensal:

In [ ]:
def calcula_prestacao_mensal(
        p: float,
        i_a: float,
        n: int,
    ) -> float:
    """
    Calcula a prestação mensal para o financiamento de um imóvel
    obedecendo ao Sistema Francês de Amortização (Tabela Price).

    Args:
    - p (float): valor total do imóvel a ser financiado.
    - i_a (float): taxa de juros anual em formato decimal.
    - n (int): prazo total de pagamento em meses.

    Returns:
    - prestacao (float): valor da prestação mensal.
    """
    i_m = (1 + i_a) ** (1/12) - 1

    numerador = i_m * (1 + i_m) ** n
    denominador = (1 + i_m) ** n - 1
    prestacao = p * (numerador / denominador)
    
    return prestacao

# Módulo de Classificação Etária (Risco Atuarial)

Na engenharia de risco financeiro, a idade do solicitante é um fator determinante para a probabilidade de inadimplência e longevidade do contrato. O modelo atuarial do IronVault estabelece faixas de riscos distintas.

A função que calcula o valor do seguro adicional ($S$) com base na idade ($i$) é modelada como uma função definida por partes:

$$S(i) = \begin{cases}
0 & \text{se } 18 \le i < 65 \\
850 & \text{se } i\ge 65
\end{cases}$$

Onde:
* **$i$** é a idade do solicitante em anos.
* **$S(i)$** é o valor em Reais (R$) do seguro de vida obrigatório adicionado à parcela.
Além disso, o sistema impõe uma barreira de entrada estrita, onde a idade mínima para contratação é de 18 anos.

Em Python, uma função traduz exatamente a regra de negócio estabelecida acima:

In [ ]:
def valida_risco_etario(idade: int) -> float:
    """
    Avalia o risco etário e determina o valor do seguro obrigatório.

    Args:
    - idade (int): idade informada pelo solicitante.

    Returns:
    - float: o valor do seguro obrigatório caso o solicitante informe
             idade entre 65 e 120 anos.

    Raises:
    - ValueError: nos casos em que o solicitante é menor de 18 anos ou
                  ultrapassa a idade limite de 120 anos.
    """
    if idade < 18 or idade > 120:
        raise ValueError("Critérios inválidos para dar prosseguimento.")

    if idade < 65:
        return 0.0
    
    return 850.00

# Módulo de Comprometimento de Renda (Risco de Crédito)

Na análise de risco de crédito, a capacidade de pagamento do solicitante é avaliada para evitar o superendividamento e a inadimplência. O IronVault utiliza um teto rígido para o comprometimento da renda mensal.
A restrição matemática para a aprovação do financiamento é modelada pela seguinte inequação:

$$P <= R \times 0.30$$

Onde:
* **$P$** é o valor da prestação mensal calculada (incluindo taxas e seguros).
* **$R$** é a renda mensal bruta comprovada do solicitante.

Se a prestação mensal ultrapassar 30% da renda ($P > R \times 0.30$), o empréstimo é considerado de alto risco e deve ser rejeitado pelo sistema.

O retorno booleano em uma função Python define a aprovação para este critério e sua assinatura e corpo encontram-se abaixo:

In [ ]:
def calcula_risco_credito(
        prestacao: float,
        renda: float,
        ) ->  bool:
    """
    Aprova um financiamento desde que a prestação mensal não exceda 30% da
    renda bruta mensal comprovada.

    Args:
    - prestacao (float): valor da prestação mensal calculada (inlcuindo taxas
    e seguros).
    - renda (float): renda bruta mensal comprovada do solicitante.

    Returns:
    - bool: o crédito é aprovado se a função retorna `True`. `False`, caso contrário.
    """
    return prestacao <= renda * 0.30

# Módulo de Score de Crédito

Na análise de risco, o score de crédito é uma métrica quantitativa que sintetiza o histórico financeiro do solicitante, variando de forma discreta em uma escala de 1 a 5.
Para fins de precificação, o modelo atua como uma correspondência direta entre o score $s$ e a taxa de juros anual $i_a(s)$, modelada por:

$$i_a(s) = \begin{cases}
\text{Rejeitado} & \text{se } s \in {1, 2} \\
0.135 & \text{se } s = 3 \\
0.095 & \text{se } s \in {4, 5}
\end{cases}$$

Onde os valores 0.135 e 0.095 representam 13.5% e 9.5% ao ano, respectivamente.

Note a correspondência entre os símbolos matemáticos e os conectivos lógicos na função Python que encapsula a abordagem matemática da regra de negócio:

In [1]:
def obter_taxa_por_score(score: int) -> float:
    """
    Determina a taxa anual de juros para os casos em que o score do solicitante
    trafega entre os índices 3 e 5. Nega o crédito para scores entre 1 e 2.

    Args:
    - score (int): valores inteiros indicadores do histórico financeiro do
    solicitante (1 a 5).

    Returns:
    - float: as taxas de juros anual de 13.5% (para score igual a 3) e 0.95% para os
    scores 4  5.

    Raises:
    - ValueError: quando o score informado está fora do intervalo (1 a 5) ou quando o
    score está entre 1 e 2.
    """
    if score < 1 or score > 5:
        raise ValueError("Critérios inválidos para cálculo da taxa.")

    if score in (4, 5):
        return 0.095
    elif score == 3:
        return 0.135
    else:
        raise ValueError("Score insuficiente. Crédito negado.")